# Day 17 — Window Functions II: LAG, LEAD, Running Totals
**Estimated time:** 60–75 min
**Datasets:** `sales_orders.csv`, `bw_sales_kpis.csv`

## Learning Objectives
- Use `LAG()` and `LEAD()` for period-over-period (MoM, YoY) comparisons
- Build running totals and moving averages using `SUM() OVER` and frame clauses
- Calculate percent-of-total contributions using `SUM() OVER ()` without partitioning

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

from analytics_bootcamp.config import RAW_DATA_DIR as DATA_DIR
# Load all CSVs
inv   = pd.read_csv(f"{DATA_DIR}/materials_inventory.csv")
sales = pd.read_csv(f"{DATA_DIR}/sales_orders.csv")
cc    = pd.read_csv(f"{DATA_DIR}/cost_center_actuals.csv")
hr    = pd.read_csv(f"{DATA_DIR}/hr_headcount.csv")
bw    = pd.read_csv(f"{DATA_DIR}/bw_sales_kpis.csv")

# Normalize column names
for df in [inv, sales, cc, hr, bw]:
    df.columns = [c.strip().upper() for c in df.columns]

# In-memory DuckDB — register pandas DataFrames as tables
con = duckdb.connect()
con.register("MATERIALS",   inv)
con.register("SALES",       sales)
con.register("COST_CENTER", cc)
con.register("HR",          hr)
con.register("BW_SALES",    bw)

def q(sql):
    return con.sql(sql).df()

# Verify
for tbl, df in [("MATERIALS",inv),("SALES",sales),("COST_CENTER",cc),("HR",hr),("BW_SALES",bw)]:
    n = q(f"SELECT COUNT(*) AS n FROM {tbl}").iloc[0,0]
    print(f"  {tbl:15s}: {n:,} rows")


In [ ]:
# -- LAG: month-over-month revenue comparison --
result = q(
    """
    WITH monthly AS (
        SELECT SUBSTR(ERDAT, 1, 7) AS ym, VKORG, SUM(NETWR) AS revenue
        FROM SALES WHERE ERDAT IS NOT NULL
        GROUP BY ym, VKORG
    )
    SELECT
        ym,
        VKORG,
        revenue,
        LAG(revenue) OVER (PARTITION BY VKORG ORDER BY ym)         AS prev_month_revenue,
        revenue - LAG(revenue) OVER (PARTITION BY VKORG ORDER BY ym) AS mom_delta,
        ROUND(
            (revenue - LAG(revenue) OVER (PARTITION BY VKORG ORDER BY ym)) * 100.0
            / NULLIF(LAG(revenue) OVER (PARTITION BY VKORG ORDER BY ym), 0)
        , 2) AS mom_pct
    FROM monthly
    ORDER BY VKORG, ym DESC
    LIMIT 24
    """
)
display(result)

In [ ]:
# -- LEAD: look ahead to next period (backlog forecasting) --
result = q(
    """
    WITH monthly AS (
        SELECT SUBSTR(ERDAT, 1, 7) AS ym, SUM(NETWR) AS revenue
        FROM SALES WHERE ERDAT IS NOT NULL GROUP BY ym
    )
    SELECT
        ym,
        revenue,
        LEAD(revenue) OVER (ORDER BY ym) AS next_month_revenue,
        ROUND(
            (LEAD(revenue) OVER (ORDER BY ym) - revenue) * 100.0
            / NULLIF(revenue, 0)
        , 2) AS forward_change_pct
    FROM monthly
    ORDER BY ym DESC
    LIMIT 18
    """
)
display(result)

In [ ]:
# -- Running total: cumulative revenue by date --
result = q(
    """
    WITH daily AS (
        SELECT ERDAT, SUM(NETWR) AS daily_revenue
        FROM SALES WHERE ERDAT IS NOT NULL
        GROUP BY ERDAT
    )
    SELECT
        ERDAT,
        ROUND(daily_revenue, 2) AS daily_revenue,
        ROUND(SUM(daily_revenue) OVER (ORDER BY ERDAT), 2) AS running_total,
        ROUND(AVG(daily_revenue) OVER (ORDER BY ERDAT ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 2) AS rolling_7d_avg
    FROM daily
    ORDER BY ERDAT DESC
    LIMIT 30
    """
)
display(result)

In [ ]:
# -- Percent of total using SUM() OVER () (no partition) --
result = q(
    """
    WITH vkorg_rev AS (
        SELECT VKORG, SUM(NETWR) AS revenue FROM SALES GROUP BY VKORG
    )
    SELECT
        VKORG,
        revenue,
        SUM(revenue) OVER ()                                AS grand_total,
        ROUND(revenue * 100.0 / SUM(revenue) OVER (), 2)   AS pct_of_total,
        ROUND(SUM(revenue) OVER (ORDER BY revenue DESC), 2) AS cumulative_revenue
    FROM vkorg_rev
    ORDER BY revenue DESC
    """
)
display(result)

In [ ]:
# -- Moving average: 3-period rolling avg on BW KPI data --
result = q(
    """
    WITH monthly_kpi AS (
        SELECT
            CAL_YEAR_MONTH,
            VKORG,
            SUM(REVENUE)       AS revenue,
            SUM(GROSS_MARGIN)  AS gross_margin
        FROM BW_SALES
        GROUP BY CAL_YEAR_MONTH, VKORG
    )
    SELECT
        CAL_YEAR_MONTH,
        VKORG,
        revenue,
        ROUND(AVG(revenue) OVER (
            PARTITION BY VKORG
            ORDER BY CAL_YEAR_MONTH
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2) AS rolling_3m_avg_rev,
        ROUND(gross_margin * 100.0 / NULLIF(revenue, 0), 2) AS gm_pct
    FROM monthly_kpi
    ORDER BY VKORG, CAL_YEAR_MONTH DESC
    LIMIT 30
    """
)
display(result)

---
## Daily Prompt

**Calculate MoM revenue change (absolute and %) for each sales org using `LAG()`. Flag months with a decline > 20% as 'ALERT'.**

```sql
-- YOUR SOLUTION
WITH monthly AS (
    SELECT SUBSTR(ERDAT, 1, 7) AS ym, VKORG, SUM(NETWR) AS revenue
    FROM SALES WHERE ERDAT IS NOT NULL
    GROUP BY ym, VKORG
)
SELECT
    ym,
    VKORG,
    revenue,
    LAG(revenue) OVER (PARTITION BY VKORG ORDER BY ym) AS prev_revenue,
    -- YOUR SOLUTION: mom_abs (absolute delta)
    -- YOUR SOLUTION: mom_pct (% change)
    -- YOUR SOLUTION: CASE WHEN flag as 'ALERT' if decline > 20%
FROM monthly
ORDER BY VKORG, ym DESC
LIMIT 30
```

> **Hint:** `ALERT` condition: `mom_pct < -20.0`
> To avoid referencing the window function alias in CASE WHEN (not allowed in SQLite), wrap in a CTE first and reference the computed column in the outer SELECT.